# 03 · Filter & Rank — run the shared multi-layer filter (`design_type="antibody"`)

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25 projects
use. **For Project 16** you map your humanized variants onto `fp.Design` objects, run the pipeline with
the **antibody** cutoffs (scRMSD ≤ 3.0, pLDDT ≥ 70, pae_interaction ≤ 12), and report survival (D3 pt 1).
We also fold in the humanization-specific axes (humanness floor + ΔΔG ceiling) as a physics-layer gate.

Run `00`–`02` first so `results/campaign.csv` exists.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Load the shared filtering pipeline

This is the cohort's shared module — improvements here are pull-requested back for everyone. Note the
`"antibody"` cutoffs; humanization variants are single-domain/Fv-style, so the antibody cutoffs apply.

In [ ]:
import filtering_pipeline as fp
import pandas as pd

print("DEFAULT_CUTOFFS:")
for k, v in fp.DEFAULT_CUTOFFS.items():
    print(" ", k, v)
print("\nUsing design_type='antibody':", fp.DEFAULT_CUTOFFS["antibody"])

## Build `fp.Design` objects from the campaign

The filter operates on `fp.Design` records. For a humanization variant the structure-confidence fields
(`plddt`, `scrmsd`, `pae_interaction`) come from an IgFold/AF2 model of the Fv — on the **mock** backend
those are not produced, so we synthesize *clearly-SYNTHETIC* placeholder confidence values from the
deterministic hash so the confidence layers have something to act on (the plumbing point). The
**humanness** and **ΔΔG proxy** are the real axes of this project: we map humanness onto `solubility`
(so Layer 3 / physics gates on a humanness floor) and stash ΔΔG + humanness + method in `extra` so they
ride along into the ranked CSV.

In [ ]:
import hashlib

def _synthetic_confidence(variant_id, method):
    """SYNTHETIC structure-confidence placeholders for the mock path (NOT real predictions).

    On a real run these come from IgFold/ImmuneBuilder + AF2 of the Fv. The over-humanized decoy is
    given deliberately weaker confidence to mimic 'over-humanization broke the fold' — purely to show
    the filter discriminating; still SYNTHETIC."""
    h = int(hashlib.sha256(str(variant_id).encode()).hexdigest(), 16)
    plddt = 72 + (h % 22)                       # 72-93
    scrmsd = round(1.0 + (h % 180) / 100.0, 3)  # 1.0-2.8 Å
    pae = 6 + (h % 7)                           # 6-12
    if method == "over_humanized_decoy":
        plddt -= 10; scrmsd += 1.2; pae += 4    # nudge the decoy toward failing (SYNTHETIC)
    return float(plddt), float(scrmsd), float(pae)

camp = pd.read_csv("results/campaign.csv")

designs = []
for _, r in camp.iterrows():
    plddt, scrmsd, pae = _synthetic_confidence(r["variant_id"], r.get("method"))
    designs.append(fp.Design(
        design_id=str(r["variant_id"]),
        sequence="",                       # full VH not needed for the confidence layers
        design_type="antibody",
        plddt=plddt, scrmsd=scrmsd, pae_interaction=pae,
        # map humanness onto solubility so Layer 3 (physics) gates on a humanness FLOOR:
        solubility=r.get("oasis_like"),
        extra={"method": r.get("method"), "oasis_like": r.get("oasis_like"),
               "t20_like": r.get("t20_like"), "ddg_kcal_mol": r.get("ddg_kcal_mol"),
               "n_framework_mutations": r.get("n_framework_mutations"),
               "n_back_mutations": r.get("n_back_mutations"),
               "synthetic": bool(r.get("synthetic", False))},
    ))
print(len(designs), "fp.Design objects built (design_type='antibody')")
print("NOTE: structure-confidence values on the mock path are SYNTHETIC placeholders.")

## Run the pipeline + report

`run_pipeline(..., design_type="antibody")` applies the layers in order with the antibody cutoffs and
returns a ranked DataFrame; `report()` prints the **survival-at-each-layer** accounting and saves the
ranked CSV + figure. We run Layers 1 + 3 here (self-consistency + physics, where physics = a humanness
floor via `min_solubility`). Layer 2 (orthogonal predictor agreement) needs a second predictor — wire
ESMFold/IgFold scRMSD in for the real run. **On mock data the survivors are SYNTHETIC** — the point is
the plumbing and the honest accounting.

In [ ]:
# Layer 3 physics gate: require humanness (mapped to `solubility`) >= a floor. Justify the floor in
# your report against real OASis/Hu-mAb distributions; 0.6 is a teaching default for the proxy.
HUMANNESS_FLOOR = 0.6

df_ranked = fp.run_pipeline(
    designs, design_type="antibody", use_layers=(1, 3),
    cutoffs={**fp.DEFAULT_CUTOFFS["antibody"]},
)
# physics_filter() uses min_solubility=-1.0 by default; re-run the physics layer explicitly with the
# humanness floor so the gate is meaningful for this project:
for d in designs:
    d.layers_passed = 0; d.notes = []
survival = {"L1": 0, "L3": 0}
for d in designs:
    if fp.self_consistency(d, fp.DEFAULT_CUTOFFS["antibody"]):
        survival["L1"] += 1
        if fp.physics_filter(d, fp.DEFAULT_CUTOFFS["antibody"], min_solubility=HUMANNESS_FLOOR):
            survival["L3"] += 1
df_ranked = fp.rank_designs(designs)
import pandas as pd
df_ranked = pd.DataFrame([fp.asdict(d) for d in df_ranked])
df_ranked.attrs["survival"] = survival
df_ranked.attrs["n_total"] = len(designs)

top = fp.report(df_ranked, top_n=15, save_prefix="results/proj16")
print("\nranked CSV -> results/proj16_ranked.csv ; survival figure -> results/proj16_survival.png")
print(f"(Layer 3 humanness floor = {HUMANNESS_FLOOR} on the OASis-like proxy.)")
top

## Hit-rate accounting (report the rate, not the cherry)

Survival = how many of the variant pool pass each layer. For humanization the meaningful "hit" is a
variant that is **human enough** (passes the humanness floor) **and** structurally self-consistent —
*and*, in the full study, has an acceptable ΔΔG. Expect the **over-humanized decoy** to fail and the
**parental control** to fail the humanness floor (it is non-human by definition) — that is the filter
behaving correctly, not a bug.

In [ ]:
n_total = df_ranked.attrs.get("n_total", len(df_ranked))
survival = df_ranked.attrs.get("survival", {})
print(f"Generated: {n_total} variants")
for layer, n in survival.items():
    print(f"  {layer}: {n} survivors ({100*n/max(n_total,1):.1f}%)")
print("\nlayers_passed distribution:")
if "layers_passed" in df_ranked:
    print(df_ranked["layers_passed"].value_counts().sort_index())

# Show how each control fared (sanity: decoy should struggle; parental should fail the humanness floor).
print("\ncontrols / methods:")
view = df_ranked.copy()
view["method"] = [e.get("method") for e in view["extra"]]
print(view[["design_id", "method", "layers_passed", "scrmsd", "plddt"]].to_string(index=False))
print("\nReminder: mock survivors are SYNTHETIC. Real survival comes from OASis/Hu-mAb + IgFold + FoldX.")

## D3 (part 1) checklist
- [ ] `results/proj16_ranked.csv` produced by the **shared** module with `design_type="antibody"`.
- [ ] Survival-at-each-layer reported (the survival figure saved), with a justified **humanness floor**.
- [ ] Mapping assumptions written down (humanness → `solubility`; ΔΔG/method stashed in `extra`).
- [ ] Controls behave sanely: over-humanized decoy struggles; parental fails the humanness floor.
- [ ] Honest hit-rate accounting; humanness + ΔΔG flagged as heuristics on the mock path.

**Next:** `04_validate.ipynb` — the humanness↔stability trade-off, Vernier back-mutations, and the
grafting-vs-resurfacing comparison.